# P1 session B - eng_Latn

Fine-tunes eng_Latn, merges the adapter into an FP16 checkpoint, and evaluates that checkpoint on BELEBELE at all three precisions.\n\n**Do not run this before session A passes.**\n\nOnly the three FT cells are produced here. The three Base cells for eng_Latn were measured in P0 and are already in `results/ALL_P0_RESULTS/tables/accuracy.csv`; re-running them would add nothing and would break the rule that a result comes from exactly one run.

**Settings: Accelerator `GPU T4 x2`, Internet `ON`.**

Nothing in this notebook configures the experiment. Every cell runs a script
from the repo; the design lives in `configs/experiment.yaml` and
`configs/p1_split_manifest.json`. If a check fails, stop and report it -- the
design is not adjusted to make a check pass.

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # branch, tag or commit SHA -- all three work

import os, subprocess, sys
SRC = "/kaggle/working/quantlang"
if not os.path.exists(SRC):
    # Full clone, then checkout. `-b` takes branch and tag names ONLY, so a
    # commit SHA there fails with exit 128 -- and `--depth 1` fetches just the
    # branch tip, which would not contain the SHA even if -b accepted one.
    subprocess.run(["git", "clone", REPO_URL, SRC], check=True)
    subprocess.run(["git", "-C", SRC, "checkout", "--quiet", REF], check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())
os.chdir(SRC); sys.path.insert(0, SRC)

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13" accelerate datasets pyyaml

# torchao is REMOVED, not upgraded. Kaggle ships torchao 0.10.0; a
# current PEFT wants >= 0.16.0, and its is_torchao_available() RAISES
# on an out-of-range version instead of returning False. PEFT probes it
# for every LoRA layer it builds, so with both installed no adapter can
# attach at all and fine-tuning cannot run.
#
# This pipeline never uses torchao -- INT8 and NF4 are both bitsandbytes.
# Upgrading it instead could pull a different torch, which is the one
# thing on Kaggle that must not move.
!pip uninstall -q -y torchao

In [ ]:
# 3. Environment probe. Raises and STOPS the notebook if this session cannot
#    run the experiment: a P100 (no INT8/NF4), or a PEFT/torchao mismatch
#    that makes LoRA attachment impossible.
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/probe_env.py", "--outdir", "/kaggle/working")

In [ ]:
# 4. The contracts still hold in THIS session.
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/freeze_p0.py")
gate("-m", "pytest", "-q")

## Fine-tune eng_Latn

One epoch of LoRA on the FP16 base, then `merge_and_unload` into a plain FP16
checkpoint. The merge is now verified to have moved the weights -- if it has
not, this cell fails rather than shipping a checkpoint that is the base model.

The training partition is trimmed to the size shared by both final-scope
languages, so each arm takes the same number of gradient steps.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/run_finetune.py", "--lang", "eng_Latn", "--outdir", "/kaggle/working/p1", "--tag", "main")

## Evaluate the FT arm

Three cells: eng_Latn x FP16 / INT8 / NF4, on the merged checkpoint, through the
same evaluator and the same frozen 900-item BELEBELE manifest that produced P0.

`--local-checkpoint` is what makes this the sanctioned path. `--ft-lang` sets
the result alias to `qwen2.5-3b-instruct-ft-eng_Latn`, so an FT cell can never
collide with or be mistaken for a Base cell, and every row records
`weights_from` and `arm`.

In [ ]:
SEED = __import__("yaml").safe_load(
    open("configs/experiment.yaml", encoding="utf-8"))["finetune"]["seeds"]["main"]
MERGED = f"/kaggle/working/p1/merged/eng_Latn__seed{SEED}"
print(MERGED)

!python scripts/run_eval.py --all-precisions --langs eng_Latn --local-checkpoint {MERGED} --ft-lang eng_Latn --outdir /kaggle/working/p1/results --tag main

## Confirm the FT arm is not the Base arm

Read straight off the written cells. `arm` must say `finetuned` and
`weights_from` must be the merged checkpoint path, not `hub`.

In [ ]:
import glob, json
for path in sorted(glob.glob("/kaggle/working/p1/results/*.meta.json")):
    m = json.load(open(path, encoding="utf-8"))
    print(f"{m['precision']:<14} {m['lang']}  acc={m['accuracy']:.4f}  "
          f"arm={m['arm']}  alias={m['model_alias']}")
    print(f"               weights_from={m['weights_from']}")
    assert m["arm"] == "finetuned", "this cell scored the BASE model"
    assert m["weights_from"] != "hub"
    assert m["n_items"] == 900, m["n_items"]
print("\nall FT cells scored the merged checkpoint on the full 900 items")

## Package

The merged FP16 checkpoint is ~5.75 GB and DERIVED -- rebuildable from the base
model plus the adapter -- so it is not shipped. The adapter, the training
metadata and every raw result are.

In [ ]:
import os, shutil
KEEP = "/kaggle/working/p1_eng_Latn_keep"
os.makedirs(KEEP, exist_ok=True)
shutil.copytree("/kaggle/working/p1/results", f"{KEEP}/results", dirs_exist_ok=True)
for d in ["adapters"]:
    if os.path.isdir(f"/kaggle/working/p1/{d}"):
        shutil.copytree(f"/kaggle/working/p1/{d}", f"{KEEP}/{d}", dirs_exist_ok=True)
for f in os.listdir("/kaggle/working/p1"):
    if f.endswith("__finetune.json"):
        shutil.copy(f"/kaggle/working/p1/{f}", KEEP)
shutil.copy("configs/p1_split_manifest.json", KEEP)

shutil.make_archive("/kaggle/working/p1_eng_Latn", "zip", KEEP)
if os.path.isdir("/kaggle/working/p1/merged"):
    shutil.rmtree("/kaggle/working/p1/merged")
    print("removed the merged checkpoint (derived, rebuildable)")
print(sorted(os.listdir("/kaggle/working")))

## Then, locally

```bash
kaggle kernels output <user>/<kernel-slug> -p results/raw/
```

`results/raw/` is append-only and is written by exactly that command. Nothing in
the repo writes to it, and `tests/test_raw_is_append_only.py` fails if something
starts to.